In [ ]:

!pip install -q transformers datasets accelerate scikit-learn sentencepiece huggingface_hub
import os, re, random, csv, json, hashlib
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from datasets import load_dataset, Dataset, concatenate_datasets
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    AutoModelForCausalLM, pipeline,
    get_linear_schedule_with_warmup
)
from collections import defaultdict
from datetime import datetime

dispozitiv = torch.device("cuda" if torch.cuda.is_available() else "cpu")
samanta = 42
random.seed(samanta)
np.random.seed(samanta)
torch.manual_seed(samanta)
if dispozitiv.type == "cuda":
    torch.cuda.manual_seed_all(samanta)
from huggingface_hub import notebook_login, HfFolder
notebook_login()
HF_TOKEN = HfFolder.get_token()

SEEN_LANGS = {"Python", "C++", "Java"}
UNSEEN_LANGS = ["Go", "PHP", "C#", "C", "JavaScript"]
subset = "A"
ds = load_dataset("DaniilOr/SemEval-2026-Task13", subset)

train_seen = ds["train"].filter(lambda x: x["language"] in SEEN_LANGS)
valid_seen = ds["validation"].filter(lambda x: x["language"] in SEEN_LANGS)

print("train_seen:", len(train_seen))
print("valid_seen:", len(valid_seen))
AUG_TOTAL_BUDGET = 2500
AUG_POOL_MAX = 1200
AUG_MAX_NEW_TOKENS = 192
AUG_TRUNC_CHARS = 3500
AUG_PIPELINE_BATCH = 4
TRANSLATOR = "bigcode/starcoder2-3b"
FALLBACK_TRANSLATOR = "bigcode/starcoder2-3b"
GEN_KWARGS = dict(
    do_sample=False,
    temperature=0.0,
    top_p=1.0,
    repetition_penalty=1.05,
    num_return_sequences=1
)

AUG_CACHE_DIR = "/content/aug_train_cache_fast"
os.makedirs(AUG_CACHE_DIR, exist_ok=True)

PAIR_CACHE_PATH = os.path.join(
    AUG_CACHE_DIR,
    f"pair_cache_budget{AUG_TOTAL_BUDGET}_pool{AUG_POOL_MAX}_tok{AUG_MAX_NEW_TOKENS}.jsonl"
)

AUG_DATASET_CACHE_PATH = os.path.join(
    AUG_CACHE_DIR,
    f"aug_train_budget{AUG_TOTAL_BUDGET}_pool{AUG_POOL_MAX}_tok{AUG_MAX_NEW_TOKENS}.parquet"
)
def extract_code(text: str) -> str:
    """
    Dacă modelul totuși scoate ```...```, extrage conținutul.
    Altfel returnează textul curățat.
    """
    blocks = re.findall(r"```(?:\w+)?\n(.*?)```", text, flags=re.DOTALL)
    if blocks:
        return blocks[-1].strip()
    return (text or "").strip()

def clean_output(s: str) -> str:
    """
    Taie răspunsuri care încep să vorbească (rar cu promptul strict, dar se întâmplă).
    """
    s = (s or "").strip()
    bad_prefixes = [
        "Explanation:", "Here is", "Sure", "I will", "Below is",
        "Target code:", "TARGET CODE:", "SOURCE CODE:", "SOURCE:"
    ]
    for bp in bad_prefixes:
        if s.startswith(bp):
            parts = s.splitlines()
            s = "\n".join(parts[1:]).strip() if len(parts) > 1 else ""
            break
    return s.strip()
def load_translator(model_name: str):
    try:
        tok_t = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
        mod_t = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
            token=HF_TOKEN
        )
        gen = pipeline("text-generation", model=mod_t, tokenizer=tok_t)
        print(" Loaded translator:", model_name)
        return gen, model_name
    except Exception as e:
        print(f" Failed to load {model_name} ({type(e).__name__}). Using fallback: {FALLBACK_TRANSLATOR}")
        tok_t = AutoTokenizer.from_pretrained(FALLBACK_TRANSLATOR, token=HF_TOKEN)
        mod_t = AutoModelForCausalLM.from_pretrained(
            FALLBACK_TRANSLATOR,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
            token=HF_TOKEN
        )
        gen = pipeline("text-generation", model=mod_t, tokenizer=tok_t)
        print(" Loaded fallback translator:", FALLBACK_TRANSLATOR)
        return gen, FALLBACK_TRANSLATOR
def make_prompt(code: str, src_lang: str, tgt_lang: str) -> str:
    code = (code or "")[:AUG_TRUNC_CHARS]
    return (
        "You are a transpiler.\n"
        f"Task: Convert {src_lang} to {tgt_lang}.\n"
        "Output rules:\n"
        "1) Output ONLY the target-language code.\n"
        "2) Do NOT use markdown fences.\n"
        "3) No comments, no explanations, no prose.\n"
        "4) Preserve logic and I/O behavior.\n"
        "5) Use only standard library.\n"
        "6) Keep it as simple as possible.\n"
        "\n"
        "SOURCE CODE:\n"
        f"{code}\n"
        "\n"
        "TARGET CODE:\n"
    )

def pair_key(code: str, src_lang: str, tgt_lang: str, translator_name: str) -> str:
    h = hashlib.md5((code or "").encode("utf-8", errors="ignore")).hexdigest()
    return f"{h}|{src_lang}|{tgt_lang}|{translator_name}"

def load_pair_cache_keys(path: str):
    keys = set()
    if not os.path.exists(path):
        return keys
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                keys.add(obj["key"])
            except Exception:
                pass
    return keys

def append_pair_cache(path: str, obj: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def build_aug_train_from_valid_budget(valid_ds):
    if os.path.exists(AUG_DATASET_CACHE_PATH):
        print("Loading cached augmented dataset:", AUG_DATASET_CACHE_PATH)
        return Dataset.from_parquet(AUG_DATASET_CACHE_PATH)
    pool = valid_ds.shuffle(seed=samanta)
    if AUG_POOL_MAX is not None:
        pool = pool.select(range(min(len(pool), AUG_POOL_MAX)))

    gen, tr_name = load_translator(TRANSLATOR)

    seen_keys = load_pair_cache_keys(PAIR_CACHE_PATH)
    print("Pair cache keys loaded:", len(seen_keys))

    rows = []
    rng = random.Random(samanta)

    i = 0
    while len(rows) < AUG_TOTAL_BUDGET:
        i += 1

        prompts = []
        metas = []
        while len(prompts) < AUG_PIPELINE_BATCH and len(rows) + len(prompts) < AUG_TOTAL_BUDGET:
            ex = pool[rng.randrange(0, len(pool))]

            src_lang = ex["language"]
            code = ex["code"]
            label = ex["label"]
            generator = ex.get("generator", None)
            tgt_lang = rng.choice(UNSEEN_LANGS)

            k = pair_key(code, src_lang, tgt_lang, tr_name)
            if k in seen_keys:
                continue

            p = make_prompt(code, src_lang, tgt_lang)
            prompts.append(p)
            metas.append((k, src_lang, tgt_lang, label, generator))
            seen_keys.add(k)

        outs = gen(
            prompts,
            max_new_tokens=AUG_MAX_NEW_TOKENS,
            **GEN_KWARGS
        )
        for out, meta, prompt in zip(outs, metas, prompts):
            k, src_lang, tgt_lang, label, generator = meta

            if isinstance(out, list):
                out_text = out[0]["generated_text"]
            else:
                out_text = out["generated_text"]
            gen_text = out_text[len(prompt):] if out_text.startswith(prompt) else out_text

            tr_code = extract_code(gen_text)
            tr_code = clean_output(tr_code)

            row = {
                "code": tr_code,
                "label": label,
                "generator": generator,
                "language": tgt_lang,
                "augmented": True,
                "src_language": src_lang,
                "translator": tr_name
            }
            rows.append(row)
            append_pair_cache(PAIR_CACHE_PATH, {"key": k, **row})

            if len(rows) % 100 == 0:
                print(f"  augmented: {len(rows)}/{AUG_TOTAL_BUDGET}")

    aug_ds = Dataset.from_list(rows)
    aug_ds.to_parquet(AUG_DATASET_CACHE_PATH)
    print("Saved augmented dataset:", AUG_DATASET_CACHE_PATH)
    return aug_ds

print("\n Building/Loading augmented train (FAST)…")
aug_train = build_aug_train_from_valid_budget(valid_seen)

print("aug_train:", len(aug_train))
print("aug_train example keys:", aug_train[0].keys())
train_final = concatenate_datasets([train_seen, aug_train]).shuffle(seed=samanta)
valid_final = valid_seen

print("train_final:", len(train_final))
print("valid_final:", len(valid_final))
_num_re = re.compile(r"\b\d+\b")
_hex_re = re.compile(r"\b0x[0-9a-fA-F]+\b")
_str_re = re.compile(r"('([^'\\]|\\.)*'|\"([^\"\\]|\\.)*\")")
_ws_re = re.compile(r"[ \t]+")

def normalize_code(code: str) -> str:
    if code is None:
        return ""
    s = code
    s = _str_re.sub("STR_LIT", s)
    s = _hex_re.sub("HEX_LIT", s)
    s = _num_re.sub("INT_LIT", s)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = _ws_re.sub(" ", s)
    s = re.sub(r"\n{3,}", "\n\n", s).strip()
    return s

def build_multiview_text(code: str) -> str:
    norm = normalize_code(code)
    return f"<RAW>\n{code}\n</RAW>\n<NORM>\n{norm}\n</NORM>"
BASE_MODEL = "microsoft/codebert-base"
tok = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
max_len = 512

def preprocess_batch(batch):
    texts = [build_multiview_text(c) for c in batch["code"]]
    enc = tok(texts, truncation=True, padding="max_length", max_length=max_len)
    enc["labels"] = batch["label"]
    enc["language"] = batch["language"]
    return enc

def prepare(ds_in):
    cols_to_remove = [c for c in ds_in.column_names if c not in ("code","label","language")]
    ds_out = ds_in.map(preprocess_batch, batched=True, remove_columns=cols_to_remove)
    ds_out.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
    return ds_out

train_proc = prepare(train_final)
valid_proc = prepare(valid_final)
valid_lang_strings = valid_final["language"]
labels_np = np.array(train_proc["labels"])
class_counts = np.bincount(labels_np, minlength=2)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[labels_np]

sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True
)

batch_size = 32
loader_train = DataLoader(train_proc, batch_size=batch_size, sampler=sampler)
loader_valid = DataLoader(valid_proc, batch_size=batch_size, shuffle=False)

print("Train class counts:", class_counts.tolist())
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    token=HF_TOKEN
).to(dispozitiv)
freeze_encoder = True
step_unfreeze = 4

if freeze_encoder:
    for p in model.roberta.parameters():
        p.requires_grad = False

layers = list(model.roberta.encoder.layer)
L = len(layers)

def unfreeze_last(n):
    n = max(0, min(L, n))
    start = L - n
    for i, layer in enumerate(layers):
        req = (i >= start)
        for p in layer.parameters():
            p.requires_grad = req

if freeze_encoder:
    unfreeze_last(step_unfreeze)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

def make_optimizer():
    enc_params = [p for p in model.roberta.parameters() if p.requires_grad]
    clf_params = list(model.classifier.parameters())
    return AdamW(
        [
            {"params": enc_params, "lr": 2e-5, "weight_decay": 0.01},
            {"params": clf_params, "lr": 1e-4, "weight_decay": 0.0},
        ]
    )

epochs = 3
total_steps = len(loader_train) * epochs

opt = make_optimizer()
sched = get_linear_schedule_with_warmup(
    opt,
    num_warmup_steps=int(total_steps * 0.06),
    num_training_steps=total_steps
)
@torch.no_grad()
def collect_logits_labels_langs():
    model.eval()
    logits_all, y_all, langs_all = [], [], []
    idx = 0
    for batch in loader_valid:
        input_ids = batch["input_ids"].to(dispozitiv)
        attn = batch["attention_mask"].to(dispozitiv)
        y = batch["labels"].to(dispozitiv)

        out = model(input_ids=input_ids, attention_mask=attn)
        logits = out.logits
        bs = logits.size(0)

        logits_all.append(logits.detach().cpu())
        y_all.extend(y.cpu().tolist())
        langs_all.extend(valid_lang_strings[idx:idx+bs])
        idx += bs

    logits_all = torch.cat(logits_all, dim=0).numpy()
    y_all = np.array(y_all)
    langs_all = np.array(langs_all)
    return logits_all, y_all, langs_all

def f1_per_language(y_true, y_pred, langs):
    by = defaultdict(lambda: {"y": [], "p": []})
    for yt, yp, l in zip(y_true, y_pred, langs):
        by[l]["y"].append(int(yt))
        by[l]["p"].append(int(yp))
    return {l: f1_score(v["y"], v["p"], average="macro") for l, v in by.items()}

def tune_threshold_for_macro_f1(logits, y_true):
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    best_f1, best_thr = -1.0, 0.5
    for thr in np.linspace(0.05, 0.95, 37):
        y_pred = (probs >= thr).astype(int)
        f1 = f1_score(y_true, y_pred, average="macro")
        if f1 > best_f1:
            best_f1, best_thr = float(f1), float(thr)
    return best_f1, best_thr

def evaluate():
    logits, y_true, langs = collect_logits_labels_langs()

    y_pred_argmax = np.argmax(logits, axis=1)
    macro_f1_argmax = float(f1_score(y_true, y_pred_argmax, average="macro"))

    best_f1, best_thr = tune_threshold_for_macro_f1(logits, y_true)
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    y_pred_thr = (probs >= best_thr).astype(int)

    return {
        "macro_f1_argmax": macro_f1_argmax,
        "macro_f1_tuned": float(best_f1),
        "best_threshold": float(best_thr),
        "f1_per_lang_tuned": f1_per_language(y_true, y_pred_thr, langs)
    }
CKPT_DIR = "/content/checkpoints_augmented_train_fast"
os.makedirs(CKPT_DIR, exist_ok=True)

CKPT_PATH = os.path.join(CKPT_DIR, "last.pt")
BEST_PATH = os.path.join(CKPT_DIR, "best.pt")
METRICS_CSV = os.path.join(CKPT_DIR, "metrics.csv")
METRICS_JSON = os.path.join(CKPT_DIR, "metrics.json")

start_epoch = 0
global_step = 0
best = -1.0
best_thr = 0.5
metrics_log = []

def save_checkpoint(epoch, global_step, best, best_thr):
    ckpt = {
        "epoch": epoch,
        "global_step": global_step,
        "best": best,
        "best_thr": best_thr,
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "sched": sched.state_dict(),
    }
    torch.save(ckpt, CKPT_PATH)

def load_checkpoint():
    global start_epoch, global_step, best, best_thr
    ckpt = torch.load(CKPT_PATH, map_location=dispozitiv)
    model.load_state_dict(ckpt["model"])
    opt.load_state_dict(ckpt["opt"])
    sched.load_state_dict(ckpt["sched"])
    start_epoch = ckpt["epoch"] + 1
    global_step = ckpt["global_step"]
    best = ckpt.get("best", -1.0)
    best_thr = ckpt.get("best_thr", 0.5)
    print(f" Resumed: next_epoch={start_epoch+1}, global_step={global_step}, best={best:.4f}, thr={best_thr:.2f}")

if not os.path.exists(METRICS_CSV):
    with open(METRICS_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([
            "timestamp","epoch","global_step","loss_mean",
            "macro_f1_argmax","macro_f1_tuned","best_threshold",
            "f1_per_lang_tuned_json"
        ])

def append_metrics(epoch, global_step, loss_mean, eval_dict):
    ts = datetime.utcnow().isoformat()
    row = {
        "timestamp": ts,
        "epoch": int(epoch),
        "global_step": int(global_step),
        "loss_mean": float(loss_mean),
        **eval_dict
    }
    metrics_log.append(row)

    with open(METRICS_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([
            ts, epoch, global_step, loss_mean,
            row["macro_f1_argmax"], row["macro_f1_tuned"], row["best_threshold"],
            json.dumps(row["f1_per_lang_tuned"])
        ])

    with open(METRICS_JSON, "w", encoding="utf-8") as f:
        json.dump(metrics_log, f, ensure_ascii=False, indent=2)

if os.path.exists(CKPT_PATH):
    load_checkpoint()
else:
    print("No checkpoint found. Start fresh.")
def rebuild_optimizer_and_sched(ep):
    global opt, sched
    opt = make_optimizer()
    remaining_steps = len(loader_train) * (epochs - ep)
    sched = get_linear_schedule_with_warmup(
        opt,
        num_warmup_steps=int(remaining_steps * 0.06),
        num_training_steps=remaining_steps
    )

for ep in range(start_epoch, epochs):
    if freeze_encoder:
        unfreeze_last((ep + 1) * step_unfreeze)
    rebuild_optimizer_and_sched(ep)

    model.train()
    losses = []

    for batch in loader_train:
        global_step += 1
        input_ids = batch["input_ids"].to(dispozitiv)
        attn = batch["attention_mask"].to(dispozitiv)
        y = batch["labels"].to(dispozitiv)

        out = model(input_ids=input_ids, attention_mask=attn)
        logits = out.logits
        loss = loss_fn(logits, y)

        losses.append(loss.item())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        opt.step()
        sched.step()
        opt.zero_grad()

    loss_mean = float(np.mean(losses)) if losses else 0.0
    eval_dict = evaluate()

    print(f"\nEpoca {ep+1}/{epochs}")
    print(f"loss_mean={loss_mean:.4f}")
    print(f" macro-F1 argmax (valid_seen) = {eval_dict['macro_f1_argmax']:.4f}")
    print(f" macro-F1 tuned  (valid_seen) = {eval_dict['macro_f1_tuned']:.4f}  @thr={eval_dict['best_threshold']:.2f}")
    print("F1 per limbaj (tuned):", {k: round(v, 4) for k, v in sorted(eval_dict["f1_per_lang_tuned"].items())})

    append_metrics(ep+1, global_step, loss_mean, eval_dict)

    if eval_dict["macro_f1_tuned"] > best:
        best = eval_dict["macro_f1_tuned"]
        best_thr = eval_dict["best_threshold"]
        torch.save(model.state_dict(), BEST_PATH)
        print(" Saved BEST:", BEST_PATH, "best tuned macro-F1:", best, "thr:", best_thr)

    save_checkpoint(ep, global_step, best, best_thr)
    print(" Saved checkpoint:", CKPT_PATH)
    print(" Metrics CSV:", METRICS_CSV)

print("\nDONE.")
print("Best tuned macro-F1:", best, "with thr:", best_thr)
print("Best model:", BEST_PATH)
print("Metrics CSV:", METRICS_CSV)
print("Metrics JSON:", METRICS_JSON)
print("Aug dataset cache:", AUG_DATASET_CACHE_PATH)
print("Pair cache:", PAIR_CACHE_PATH)
